In [13]:
from IPython.core.debugger import prompt
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict

In [14]:
load_dotenv()

True

In [15]:
# Model
model = ChatGoogleGenerativeAI(model='gemini-3-flash-preview')

In [16]:
class BlogState(TypedDict):
    title: str
    content: str
    outline: str

In [17]:
def create_outline(state: BlogState):
    title = state['title']

    prompt = f'generate a detailed outline for a blog on the topic:\n{title}'
    outline = model.invoke(prompt).content
    state['outline'] = outline

    return state

In [18]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'generate a detailed blog on the topic: \n{title} using the following outline: \n{outline}'
    content = model.invoke(prompt).content
    state['content'] = content

    return state

In [19]:
graph = StateGraph(BlogState)

In [20]:
# node
graph.add_node('create_outline_node', create_outline)
graph.add_node('create_blog_node', create_blog)

In [21]:
# Edges
graph.add_edge(START, 'create_outline_node')
graph.add_edge('create_outline_node', 'create_blog_node')
graph.add_edge('create_blog_node', END)

In [22]:
# Compile
workflow = graph.compile()

In [24]:
initial_state = {
    'title': 'Rise of AI in India',
}

result = workflow.invoke(initial_state)

print(result)
print(result['title'])
print(result['content'][0]['text'])
print(result['outline'][0]['text'])

{'title': 'Rise of AI in India', 'content': [{'type': 'text', 'text': '# The Silicon Renaissance: Mapping the Rise of AI in India\n\nIn early 2024, a significant milestone echoed through the global tech corridors: the Indian government approved the ambitious **IndiaAI Mission** with a staggering budget of ₹10,372 crore. Shortly after, Krutrim became the country\'s first AI startup to achieve unicorn status. These aren\'t just isolated success stories; they are the drumbeats of a massive technological shift.\n\nFor decades, India was celebrated as the world’s "back office," the reliable engine room for global IT services. Today, that narrative is being rewritten. India is rapidly evolving from a service provider into the world’s **"AI laboratory,"** where innovative solutions are being forged at a scale seen nowhere else on Earth.\n\nWith a unique combination of massive data scale, a booming talent pool, and proactive government intervention, India is not just participating in the AI re